# Tuần 3–4: ViT5 inference, ROUGE và chuẩn bị dữ liệu thô

Notebook này chạy toàn bộ phần thực hành Tuần 3–4 theo đúng pipeline của project:

1. Đọc và kiểm tra dữ liệu thô cho Tuần 4.
2. Chuẩn hóa tối thiểu, loại cặp lỗi/trùng chính xác và lưu audit.
3. Tải checkpoint ViT5 đã pre-train để sinh tóm tắt cho Tuần 3.
4. Tính ROUGE-1, ROUGE-2, ROUGE-L và xem từng prediction.

> Lần chạy đầu tiên cần Internet để tải checkpoint `VietAI/vit5-base-vietnews-summarization`. GPU CUDA là tùy chọn; notebook tự chọn GPU nếu khả dụng.

## 1. Thiết lập môi trường

Khi chạy local, mở notebook từ thư mục `tuan 3-4/summarization`. Khi chạy bằng Colab, cell bên dưới sẽ tự clone repository vào `/content/thuc-tap` nếu project chưa có trên runtime. Lần đầu sẽ cần Internet.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_SUBDIR = Path('tuan 3-4') / 'summarization'
IN_COLAB = 'google.colab' in sys.modules

def is_project(path: Path) -> bool:
    return (path / 'src' / 'week34_summarization').is_dir()

# Runtime Colab không có các file trên máy local. Tải source một lần cho mỗi runtime mới.
COLAB_WORKSPACE = Path('/content/thuc-tap')
if IN_COLAB and not is_project(COLAB_WORKSPACE / PROJECT_SUBDIR):
    if COLAB_WORKSPACE.exists():
        raise RuntimeError(
            f'{COLAB_WORKSPACE} đã tồn tại nhưng không chứa project hợp lệ. ' 
            'Hãy xóa thư mục này trên runtime rồi chạy lại cell.'
        )
    subprocess.run(
        ['git', 'clone', 'https://github.com/dungcony/text-sumarization.git', str(COLAB_WORKSPACE)],
        check=True,
    )

# Hỗ trợ cả local (mở tại root hoặc notebooks/) và Colab.
cwd = Path.cwd().resolve()
candidates = (
    cwd,
    cwd.parent,
    cwd / PROJECT_SUBDIR,
    COLAB_WORKSPACE / PROJECT_SUBDIR,
)
PROJECT_ROOT = next((path for path in candidates if is_project(path)), None)
if PROJECT_ROOT is None:
    tried = '\n'.join(f'  - {path}' for path in candidates)
    raise FileNotFoundError(f'Không tìm thấy src/. Đã kiểm tra:\n{tried}')

WORKSPACE_ROOT = PROJECT_ROOT.parent.parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('PROJECT_ROOT :', PROJECT_ROOT)
print('WORKSPACE_ROOT:', WORKSPACE_ROOT)

In [ ]:
# Colab là runtime mới nên cần cài package của project và dependency.
# Local: bỏ dấu # ở dòng dưới nếu môi trường chưa được cài.
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
# else:
#     subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)

import torch
import transformers
import pandas
import rouge_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Thiết bị sẽ dùng:', DEVICE)

## 2. Tuần 4 — Chuẩn bị và audit dữ liệu thô

Nguồn ở workspace có 1.000 cặp `Document`–`Summary`. Bước dưới đây chỉ chuẩn hóa Unicode/HTML/khoảng trắng, kiểm tra độ dài và loại cặp trùng chính xác. Chưa chia train/validation/test, chưa chunking và chưa fine-tune.

In [ ]:
from week34_summarization.data import (
    read_rows, records_from_rows, write_records_csv, write_json,
)

RAW_SOURCE = WORKSPACE_ROOT / 'data' / 'vietnamese-summarization' / 'train' / 'bio_medicine.csv'
RAW_OUTPUT = PROJECT_ROOT / 'data' / 'raw' / 'vietnews_medical_raw_1000.csv'
AUDIT_OUTPUT = RAW_OUTPUT.with_name(f'{RAW_OUTPUT.stem}_audit.json')

rows = read_rows(RAW_SOURCE)
records, audit = records_from_rows(
    rows,
    dataset_name='vietnews-medical',
    min_article_words=10,
    min_summary_words=3,
    keep_at_most=1000,
)
write_records_csv(records, RAW_OUTPUT)
write_json(AUDIT_OUTPUT, audit)

print(f'Nguồn: {len(rows)} bản ghi')
print(f'Đầu ra: {len(records)} cặp hợp lệ')
print('Loại:', audit['rejected_by_reason'])
audit

In [ ]:
import pandas as pd

raw_frame = pd.read_csv(RAW_OUTPUT)
raw_frame[['id', 'article', 'summary', 'source_dataset']].head(3)

## 3. Tuần 3 — Tải ViT5 và sinh tóm tắt

Cell này chạy smoke test trên 5 mẫu có reference. Giữ số mẫu nhỏ để kiểm tra pipeline trước; muốn đánh giá nghiêm túc, thay bằng một **test set cố định** chưa xuất hiện trong fine-tuning và tăng `MAX_SAMPLES`.

In [ ]:
from week34_summarization.model import generate_summaries
from week34_summarization.metrics import compute_rouge, compression_statistics
from week34_summarization.data import records_from_rows, read_rows

MODEL_NAME = 'VietAI/vit5-base-vietnews-summarization'
SAMPLES_PATH = WORKSPACE_ROOT / 'data' / 'summarization_samples.json'
MAX_SAMPLES = 5

sample_records, sample_audit = records_from_rows(
    read_rows(SAMPLES_PATH), keep_at_most=MAX_SAMPLES
)
articles = [record.article for record in sample_records]
references = [record.summary for record in sample_records]

predictions, used_device = generate_summaries(
    articles,
    model_name=MODEL_NAME,
    device='auto',
    prefix='summarize: ',
    batch_size=1,
    max_source_length=384,
    max_new_tokens=64,
    min_new_tokens=8,
    num_beams=2,
    no_repeat_ngram_size=3,
)

print(f'Đã chạy {len(predictions)} mẫu trên {used_device}.')

In [ ]:
rouge = compute_rouge(predictions, references)
compression = compression_statistics(articles, predictions)

score_table = pd.DataFrame(
    {metric: values for metric, values in rouge.items()}
).T.rename(columns={'precision': 'Precision', 'recall': 'Recall', 'f1': 'F1'})
display(score_table)
print('Thống kê độ dài:', compression)

In [ ]:
prediction_frame = pd.DataFrame(
    {
        'id': [record.id for record in sample_records],
        'article': articles,
        'reference': references,
        'prediction': predictions,
    }
)
pd.set_option('display.max_colwidth', 180)
prediction_frame[['id', 'reference', 'prediction']]

## 4. Lưu kết quả tái lập được

Lưu prediction, cấu hình generation và ROUGE giúp đối chiếu chính xác với checkpoint sau fine-tuning ở Tuần 5–6. Cell này không ghi đè file kết quả của báo cáo; nó tạo một file notebook riêng.

In [ ]:
import json
from datetime import datetime, timezone

NOTEBOOK_RESULT = PROJECT_ROOT / 'results' / 'notebook_vit5_sample_5.json'
payload = {
    'run': {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model': MODEL_NAME,
        'device': used_device,
        'number_of_examples': len(sample_records),
    },
    'generation': {
        'prefix': 'summarize: ', 'max_source_length': 384,
        'max_new_tokens': 64, 'min_new_tokens': 8,
        'num_beams': 2, 'no_repeat_ngram_size': 3,
    },
    'metrics': {'rouge': rouge, 'compression': compression},
    'predictions': prediction_frame.to_dict(orient='records'),
}
NOTEBOOK_RESULT.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu:', NOTEBOOK_RESULT)

## Kết luận và bàn giao

- Tuần 3 hoàn thành baseline pretrained ViT5 và ROUGE. Luôn đọc prediction để phát hiện hallucination; ROUGE không thay thế kiểm tra tính đúng đắn.
- Tuần 4 hoàn thành dữ liệu raw có schema chuẩn và audit.
- Tuần 5–6 nhận file raw này để kiểm tra near-duplicate, chia train/validation/test, xử lý văn bản dài và fine-tune. Không dùng 5 mẫu smoke test này làm tập test cuối cùng.